# Lesson 5, one call at a time

Lesson 5 asked how far a measured score can be trusted, and spent its three hours on
the tools that answer it: a validation set, cross-validation and its spread, stratified
and grouped folds, bias and variance, training against validation error, leakage that
survives cross-validation, `GridSearchCV`, nested cross-validation, a threshold chosen
on a validation set, and seeds. This notebook takes each step on its
own, on ten rows where a pencil can follow it and on the lesson's own 800 drives where
the lesson's numbers come from, so that the slide's number and the call that makes it
sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. Notebook `00_the_scikit_learn_api.ipynb` explains `fit`,
`predict`, `predict_proba`, `random_state` and `Pipeline`; this one assumes them.

Every atom names the slide it comes from, by title, and lesson 5's notebooks treat
each idea at full scale. The whole notebook runs in about two minutes; atoms 12 to 14
take most of it, because they repeat notebook 3's search.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold,
                                     RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Lesson 5's data: the 800 drives (lesson 4's fleet, cut down) and the energy curve.
sys.path.insert(0, str(Path.cwd().parent / "Lessons" / "05_experimental_methodology"
                       / "Notebooks"))
from methodology_data import load_fleet, sample_energy, true_curve

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)


def fresh_model(C=1.0):
    # notebook 1's model: standardise, then logistic regression
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000, C=C))


X, y = load_fleet(800)
print(f"{len(X)} drives, {int(y.sum())} failures, columns: {list(X.columns)}")

800 drives, 29 failures, columns: ['reallocated_sectors', 'spin_retry_count', 'read_error_rate', 'temperature_c', 'power_on_hours', 'seek_error_rate']


## 1. Choosing C on a validation set: the winner is a number

Slide 10's left recipe on the 800 drives, with slide 52's split: 200 drives set aside
as the test set, 150 as the validation set, 450 to train. Each candidate C gets one fit
on the 450 and one AUC on the 150; the best AUC picks a **value of C**, and that value
is refitted on all 600 non-test drives before the test set is scored, once.

*From:* slides *Choosing a hyperparameter on a validation set: two recipes* and *The
same three models: who wins, and how they score*; handout section 3.

In [2]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.25,
                                                  random_state=0, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_rest, y_rest, test_size=0.25,
                                                      random_state=0, stratify=y_rest)
print(f"train {len(X_train)}, validation {len(X_valid)} ({int(y_valid.sum())} failures), "
      f"test {len(X_test)} ({int(y_test.sum())} failures)")

train 450, validation 150 (5 failures), test 200 (7 failures)


### Your prediction, before you run the next cell

How many fits before the test set is touched? Will the three validation AUCs differ by
more than a few thousandths? After the refit on 600 drives, will the test AUC equal the
winner's validation AUC?

In [3]:
valid_auc = {}
for C in (0.01, 1, 100):
    candidate = fresh_model(C).fit(X_train, y_train)          # one fit per candidate
    valid_auc[C] = roc_auc_score(y_valid, candidate.predict_proba(X_valid)[:, 1])
    print(f"C = {C:>6}: fit on {len(X_train)}, AUC on the {len(X_valid)} validation "
          f"drives {valid_auc[C]:.4f}")

ranked = sorted(valid_auc, key=valid_auc.get, reverse=True)
best_C = ranked[0]
print(f"\nwinner: C = {best_C}, a number; the three fitted models are discarded")
print(f"margin over the runner-up (C = {ranked[1]}): "
      f"{valid_auc[ranked[0]]:.4f} - {valid_auc[ranked[1]]:.4f} = "
      f"{valid_auc[ranked[0]] - valid_auc[ranked[1]]:.4f}")

final = fresh_model(best_C).fit(X_rest, y_rest)               # fit number 4, on all 600
test_auc = roc_auc_score(y_test, final.predict_proba(X_test)[:, 1])
print(f"refit with C = {best_C} on {len(X_rest)} drives (fit 4); "
      f"test AUC, scored once: {test_auc:.4f}")

C =   0.01: fit on 450, AUC on the 150 validation drives 0.9793
C =      1: fit on 450, AUC on the 150 validation drives 0.9779
C =    100: fit on 450, AUC on the 150 validation drives 0.9779

winner: C = 0.01, a number; the three fitted models are discarded
margin over the runner-up (C = 1): 0.9793 - 0.9779 = 0.0014
refit with C = 0.01 on 600 drives (fit 4); test AUC, scored once: 0.9571


**What to look at.** Four fits before the test set is touched: three candidates on the
450, one refit on the 600. On the 150 validation drives C = 0.01 scores 0.9793 and
C = 1 and C = 100 tie at 0.9779: the winner wins by 0.9793 − 0.9779 = 0.0014, the kind
of hair-thin margin slide *The same three models: who wins, and how they score* showed
on 200 splits. What wins is the number 0.01; the three fitted models are thrown away,
and the refit on all 600 non-test drives is a new model. Its test AUC, 0.9571, is not
the winner's validation AUC: it is a fresh measurement on 200 drives no choice has
seen, and the one to report.

**Try changing.** add C = 0.1 and C = 10 to the candidates: does the winner change, and
by how much does it win?

## 2. Cross-validation on ten days: who trains, who is held out

Five-fold cross-validation cuts the rows into five folds and runs five rounds: each
round fits a fresh model on four folds and scores it on the fifth. On ten days each fold
is two days, small enough to see every round. The score here is the mean squared error
(MSE); scikit-learn's scorers are "higher is better", so it reports the MSE with a minus
sign, `neg_mean_squared_error`.

*From:* slide *k-fold cross-validation: the validation set, rotated*; notebook 1, section 4.

In [4]:
temperature, energy = sample_energy(10, np.random.default_rng(0))
days = pd.DataFrame({"temperature (°C)": temperature.round(1),
                     "energy (kWh)": energy.round(1)})
print(days.to_string())

curve = make_pipeline(PolynomialFeatures(2), LinearRegression())   # degree 2
folds = KFold(n_splits=5, shuffle=True, random_state=0)

   temperature (°C)  energy (kWh)
0              22.4         229.3
1               6.6         324.3
2              -3.2         428.2
3              -4.3         497.9
4              30.0         302.6
5              34.2         378.2
6              21.1         223.1
7              26.4         275.2
8              18.4         237.2
9              35.2         432.8


### Your prediction, before you run the next cell

How many times is a model fitted, and how many times is each day used for scoring?
Will `cross_val_score` give five MSEs, or five MSEs with a minus sign?

In [5]:
T = temperature.reshape(-1, 1)
by_hand = []
for round_number, (train_rows, held_out) in enumerate(folds.split(T), 1):
    model = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(
        T[train_rows], energy[train_rows])
    mse = mean_squared_error(energy[held_out], model.predict(T[held_out]))
    by_hand.append(mse)
    print(f"round {round_number}: train on days {sorted(train_rows)}, "
          f"score days {sorted(held_out)}: MSE {mse:,.1f}")

by_hand = np.array(by_hand)
print(f"\nmean by hand: {' + '.join(f'{v:,.1f}' for v in by_hand)} = "
      f"{by_hand.sum():,.1f}; ÷ 5 = {by_hand.sum() / 5:,.1f}")
library = cross_val_score(curve, T, energy, cv=folds, scoring="neg_mean_squared_error")
print("cross_val_score:", library.round(1))
print("same five numbers, sign flipped:", np.allclose(-library, by_hand))

round 1: train on days [0, 1, 3, 4, 5, 6, 7, 9], score days [2, 8]: MSE 1,569.0
round 2: train on days [0, 1, 2, 3, 5, 6, 7, 8], score days [4, 9]: MSE 1,033.9
round 3: train on days [0, 2, 3, 4, 5, 7, 8, 9], score days [1, 6]: MSE 965.7
round 4: train on days [0, 1, 2, 4, 5, 6, 8, 9], score days [3, 7]: MSE 399.5
round 5: train on days [1, 2, 3, 4, 6, 7, 8, 9], score days [0, 5]: MSE 274.6

mean by hand: 1,569.0 + 1,033.9 + 965.7 + 399.5 + 274.6 = 4,242.6; ÷ 5 = 848.5
cross_val_score: [-1569.  -1033.9  -965.7  -399.5  -274.6]
same five numbers, sign flipped: True


**What to look at.** Five fits, one per round, and every day is scored exactly once,
in the round where it is held out: days 2 and 8 in round 1, days 4 and 9 in round 2, and
so on. The five MSEs run from 274.6 to 1,569.0 kWh²: on two days, one awkward day moves a
round a long way. Their mean, 4,242.6 ÷ 5 = 848.5 kWh², is the cross-validated MSE.
`cross_val_score` returns the same five numbers with a minus sign, because its scorers
are "higher is better" and −1,569.0 must count as worse than −274.6. Flip the sign
before reporting.

**Try changing.** set `random_state=1` in `KFold`: which days go together now, and how much does the
mean move?

## 3. The same on the 800 drives: five round AUCs and their mean

On the fleet the score is the area under the receiver operating characteristic
(ROC) curve, the AUC of lesson 4, and the folds are
stratified (atom 6 shows why). The lesson's number, 0.9514, is the mean of five round
AUCs.

*From:* slide *What it actually estimates*; notebook 1, section 4.

### Your prediction, before you run the next cell

Will the five round AUCs agree to the second decimal? Will the loop by hand and
`cross_val_score` give the same five numbers?

In [6]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
round_auc = []
for train_rows, held_out in folds.split(X, y):
    model = fresh_model().fit(X.iloc[train_rows], y.iloc[train_rows])
    round_auc.append(roc_auc_score(y.iloc[held_out],
                                   model.predict_proba(X.iloc[held_out])[:, 1]))
round_auc = np.array(round_auc)

print("five round AUCs:", round_auc.round(3))
print(f"mean: {' + '.join(f'{v:.3f}' for v in round_auc)} = {round_auc.sum():.3f};"
      f" ÷ 5 = {round_auc.mean():.4f}")
library = cross_val_score(fresh_model(), X, y, cv=folds, scoring="roc_auc")
print("cross_val_score gives the same:", np.allclose(library, round_auc))

five round AUCs: [0.929 0.958 0.984 0.939 0.947]
mean: 0.929 + 0.958 + 0.984 + 0.939 + 0.947 = 4.757; ÷ 5 = 0.9514
cross_val_score gives the same: True


**What to look at.** The five AUCs run from 0.929 to 0.984: they agree on the first
decimal, not on the second. Their sum is 4.757, and 4.757 ÷ 5 = 0.9514, the number on
slide *What it actually estimates*. The loop and `cross_val_score` give the same five
numbers because they use the same folds and the same recipe; `cross_val_score` is that
loop, written once.

**Try changing.** use `KFold(5, shuffle=True, random_state=0)` instead: do the five AUCs move?

## 4. Five rounds, five different models

Each round fits its own model, and the model you ship is a sixth, refitted on all
800 drives. The 0.9514 measures the recipe, not any one of the six.

*From:* slide *Five rounds, five different models; you ship a sixth, fitted on all 800*; notebook 1, section 4.

### Your prediction, before you run the next cell

Will the five round models have the same coefficients? Will the final model's
coefficients sit among theirs?

In [7]:
round_models = [fresh_model().fit(X.iloc[train_rows], y.iloc[train_rows])
                for train_rows, _ in folds.split(X, y)]
final_model = fresh_model().fit(X, y)

coefficients = pd.DataFrame(
    [m[-1].coef_[0] for m in round_models] + [final_model[-1].coef_[0]],
    index=[f"round {r}" for r in range(1, 6)] + ["final, all 800"],
    columns=X.columns)
print("coefficients on the standardised features:")
print(coefficients.round(3).to_string())

flags = np.array([m.predict_proba(X)[:, 1] >= 0.5 for m in round_models])
disagree = int((flags.min(axis=0) != flags.max(axis=0)).sum())
print(f"\ndrives on which the five round models disagree at 0.5: {disagree} of 800")

coefficients on the standardised features:
                reallocated_sectors  spin_retry_count  read_error_rate  temperature_c  power_on_hours  seek_error_rate
round 1                       1.723             1.152            0.494          0.442           0.070           -0.163
round 2                       1.844             1.044            0.545          0.236           0.353            0.024
round 3                       1.819             0.839            0.637          0.281           0.010           -0.069
round 4                       1.881             0.955            0.588          0.175           0.228           -0.178
round 5                       1.822             0.948            0.422          0.435           0.072            0.057
final, all 800                1.848             0.987            0.553          0.317           0.140           -0.063

drives on which the five round models disagree at 0.5: 8 of 800


**What to look at.** No two rows are the same. `reallocated_sectors` stays between
1.72 and 1.88, but `power_on_hours` runs from 0.010 to 0.353, and `seek_error_rate`
changes sign, from −0.178 to +0.057. The final model is a sixth set of coefficients,
among the five and equal to none, and at a threshold of 0.5 the five round models
disagree about 8 of the 800 drives. So 0.9514 measures the recipe, refitted five times;
the model you ship was never scored.

**Try changing.** print the intercepts too, `m[-1].intercept_`: do they move as much as the slopes?

## 5. The spread: one standard deviation, by hand, and the report line

The lesson's report gives a mean and a **spread**: the standard deviation $s$ of
the five round scores. $s = \sqrt{\sum (a_i - \bar a)^2 / 5}$; NumPy's `std()` divides
by 5, and `std(ddof=1)` by 4, which gives a slightly larger figure. Say which.

*From:* slides *Report the spread, but not as a confidence interval* and *What to put
in a report*; notebook 1, section 4.

### Your prediction, before you run the next cell

Roughly how large will $s$ be: 0.002, 0.02 or 0.2? And which is larger, `std()` or
`std(ddof=1)`?

In [8]:
mean = round_auc.sum() / 5
deviations = round_auc - mean
print(f"{'round AUC':>10}{'deviation':>11}{'squared':>11}")
for a, d in zip(round_auc, deviations):
    print(f"{a:>10.4f}{d:>+11.4f}{d ** 2:>11.6f}")
total = (deviations ** 2).sum()
print(f"\nsum of squares {total:.6f}; ÷ 5 = {total / 5:.6f}; "
      f"square root s = {np.sqrt(total / 5):.4f}")
print(f"np.std: {round_auc.std():.4f}    np.std(ddof=1): {round_auc.std(ddof=1):.4f}")
print(f'\nthe report line: "AUC {mean:.3f} ± {round_auc.std():.3f} (standard deviation '
      f'over 5 folds), stratified 5-fold, seed 0"')
print(f"not the error bar: s ÷ √5 = {round_auc.std() / np.sqrt(5):.4f} is too small "
      f"(slide 17: the real spread is about twice it)")

 round AUC  deviation    squared
    0.9290    -0.0224   0.000500
    0.9578    +0.0064   0.000041
    0.9838    +0.0324   0.001048
    0.9394    -0.0120   0.000144
    0.9470    -0.0044   0.000020

sum of squares 0.001753; ÷ 5 = 0.000351; square root s = 0.0187
np.std: 0.0187    np.std(ddof=1): 0.0209

the report line: "AUC 0.951 ± 0.019 (standard deviation over 5 folds), stratified 5-fold, seed 0"
not the error bar: s ÷ √5 = 0.0084 is too small (slide 17: the real spread is about twice it)


**What to look at.** $s = 0.0187$: the spread lives in the second decimal. The
squares add to 0.001753; divided by 5 that is 0.000351, and its square root is 0.0187,
which is what `std()` gives. `std(ddof=1)` divides by 4 and gives 0.0209; the lesson
uses `std()`, and the report should say which. The report line is the one handout section 9.1
calls reproducible and honest. What it is not: $s ÷ \sqrt{5} =
0.0084$ is too small an error bar, because the five training sets share most of their
rows; across 300 new fleets the real spread of the mean was 0.0165, about twice
(slide *300 new fleets: twice the naive spread; compare the widths*).

**Try changing.** change the seed of `folds` to 3 and recompute: does $s$ stay near the same size?

## 6. Stratification, on twenty rows with five positives

`KFold` cuts the rows without looking at the label; `StratifiedKFold` deals each
class out evenly. With a rare class that matters: a fold with no positive cannot even
compute an AUC.

*From:* slide *Stratify when a class is rare*; notebook 1, section 5.

In [9]:
label = np.array([1] * 5 + [0] * 15)            # 20 rows, 5 positives
rows = np.zeros((20, 1))                           # the features do not matter here
print("labels:", label)

labels: [1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


### Your prediction, before you run the next cell

With five folds of four rows, how many positives will each fold get from `KFold`, and
how many from `StratifiedKFold`?

In [10]:
for name, splitter in (("KFold", KFold(5, shuffle=True, random_state=0)),
                       ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=0))):
    counts = [int(label[held_out].sum()) for _, held_out in splitter.split(rows, label)]
    print(f"{name:>16}: positives per fold {counts}")

print("\nthe 800 drives at seed 3, failures per fold of 160 and the fold AUCs:")
for name, splitter in (("KFold", KFold(5, shuffle=True, random_state=3)),
                       ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=3))):
    counts = [int(y.iloc[held_out].sum()) for _, held_out in splitter.split(X, y)]
    aucs = cross_val_score(fresh_model(), X, y, cv=splitter, scoring="roc_auc")
    print(f"{name:>16}: {counts}  AUC {aucs.round(3)}")

           KFold: positives per fold [1, 0, 2, 0, 2]
 StratifiedKFold: positives per fold [1, 1, 1, 1, 1]

the 800 drives at seed 3, failures per fold of 160 and the fold AUCs:
           KFold: [7, 1, 7, 5, 9]  AUC [0.979 1.    0.942 0.965 0.943]
 StratifiedKFold: [5, 6, 6, 6, 6]  AUC [0.983 0.969 0.902 0.945 0.969]


**What to look at.** On the twenty rows `KFold` gives 1, 0, 2, 0 and 2 positives: two
folds with no positive at all, where an AUC cannot even be computed. `StratifiedKFold`
gives exactly one each, which is all that five positives allow. On the 800 drives at
seed 3, plain `KFold` deals 7, 1, 7, 5 and 9 failures to the five folds of 160, and the
fold with a single failure scores a perfect 1.000: one failing drive ranked above 159
healthy ones decides the whole AUC. Stratified, the folds hold 5, 6, 6, 6 and 6, and no
fold is perfect.

**Try changing.** try `KFold` with `random_state` from 0 to 9 on the twenty rows: how often does a fold
get no positive at all?

## 7. Repeated cross-validation: the seed moves the score less

Changing only the seed of the folds moves a 5-fold score a little.
`RepeatedStratifiedKFold(5, n_repeats=10)` runs ten 5-fold cross-validations with
different shuffles and averages all fifty rounds.

*From:* slides *Change only the seed, 40 times* and *How many folds?* (both optional);
notebook 1, section 6.

### Your prediction, before you run the next cell

Over 40 seeds, will repeated 5-fold spread less than plain 5-fold? By about how
much: half, a third, a tenth?

In [11]:
plain, repeated = [], []
for seed in range(40):
    plain.append(cross_val_score(
        fresh_model(), X, y, scoring="roc_auc",
        cv=StratifiedKFold(5, shuffle=True, random_state=seed)).mean())
    repeated.append(cross_val_score(
        fresh_model(), X, y, scoring="roc_auc",
        cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=seed)).mean())
plain, repeated = np.array(plain), np.array(repeated)
for name, v in (("5-fold", plain), ("5-fold, repeated 10 times", repeated)):
    print(f"{name:>26}: mean {v.mean():.4f}, from {v.min():.4f} to {v.max():.4f}, "
          f"standard deviation {v.std():.4f}")
print(f"\nratio of the standard deviations: {plain.std() / repeated.std():.1f}")

                    5-fold: mean 0.9532, from 0.9439 to 0.9588, standard deviation 0.0032
 5-fold, repeated 10 times: mean 0.9533, from 0.9509 to 0.9550, standard deviation 0.0009

ratio of the standard deviations: 3.8


**What to look at.** Over 40 seeds, plain 5-fold moves from 0.9439 to 0.9588, a
standard deviation of 0.0032; repeated ten times, from 0.9509 to 0.9550, a standard
deviation of 0.0009, about a quarter, for ten times the fits (50 per estimate). The
means barely differ, 0.9532 and 0.9533. Repetition removes the wobble that comes from
the seed of the folds; it does nothing about the wobble that comes from having these
800 drives and not others, which is what slide *300 new fleets: twice the naive spread; compare the widths* measured.

**Try changing.** try `n_repeats=3`: how much of the gain is left?

## 8. Ten readings per drive: keep each drive in one fold

When every drive contributes ten nearly identical rows, a shuffled split puts some of
a drive's readings in training and the others in validation: the model can look the
answer up. `GroupKFold` keeps all of a drive's rows in one fold. The simulation is the
handout's (section 4.6): each drive's values plus noise of 5% of each column's spread.

*From:* slides *When the rows are not independent* and *Split by the thing you must
generalise to*; handout section 4.6.

In [12]:
readings = 10
noise_rng = np.random.default_rng(0)
X_read = pd.DataFrame(np.repeat(X.values, readings, axis=0)
                      + noise_rng.normal(0, 0.05, (len(X) * readings, X.shape[1]))
                      * X.std().values, columns=X.columns)
y_read = pd.Series(np.repeat(y.values, readings))
drive_id = np.repeat(np.arange(len(X)), readings)
print(f"{len(X_read):,} rows, {len(np.unique(drive_id))} drives")

# drive_id: one value per row, the drive each reading came from. Not a feature:
# it is passed apart, as groups=, and only tells GroupKFold which rows go together.
preview = X_read.assign(failed=y_read, drive_id=drive_id)
print(preview.iloc[[0, 1, 9, 10, 11, len(preview) - 1]].round(2).to_string())
print()

grouped = GroupKFold(5)
for fold, (_, held_out) in enumerate(grouped.split(X_read, y_read, groups=drive_id), 1):
    drives = np.unique(drive_id[held_out])
    print(f"fold {fold}: {len(held_out):,} rows from {len(drives)} drives, "
          f"e.g. drives {drives[:5]}")
all_drives = [set(drive_id[h]) for _, h in grouped.split(X_read, y_read, groups=drive_id)]
print("a drive in two folds:", any(a & b for i, a in enumerate(all_drives)
                                   for b in all_drives[i + 1:]))

8,000 rows, 800 drives
      reallocated_sectors  spin_retry_count  read_error_rate  temperature_c  power_on_hours  seek_error_rate  failed  drive_id
0                    1.39              1.10              0.0          36.02        18860.88             0.01       0         0
1                    1.43              1.11              0.0          35.73        18818.67             0.01       0         0
9                    1.35              1.10              0.0          36.15        18548.42             0.01       0         0
10                   1.60             -0.02              0.0          34.39        24430.52             0.00       1         1
11                   1.66              0.02              0.0          34.03        24297.07             0.00       1         1
7999                 1.39              0.69              0.0          37.48        32840.16             0.00       0       799

fold 1: 1,600 rows from 160 drives, e.g. drives [ 4  9 14 19 24]
fold 2: 1,600 rows fro

### Your prediction, before you run the next cell

A 1-nearest-neighbour classifier copies the label of the closest training row. What
AUC will it get with shuffled stratified folds, and what with `GroupKFold`?

In [13]:
nearest = make_pipeline(StandardScaler(), KNeighborsClassifier(1))
shuffled = cross_val_score(nearest, X_read, y_read, scoring="roc_auc",
                           cv=StratifiedKFold(5, shuffle=True, random_state=0))
kept_together = cross_val_score(nearest, X_read, y_read, scoring="roc_auc",
                                cv=GroupKFold(5), groups=drive_id)
print(f"rows shuffled:              AUC {shuffled.mean():.3f}")
print(f"each drive's rows together: AUC {kept_together.mean():.3f}")

rows shuffled:              AUC 1.000
each drive's rows together: AUC 0.620


**What to look at.** First the table: rows 0 to 9 are the ten readings of drive 0, nearly the same values and the same label; row 10 starts drive 1, and the last row belongs to drive 799. That last column is what `groups=drive_id` passes. `GroupKFold` puts all ten readings of a drive in one fold: 1,600
rows from 160 drives per fold, and no drive in two folds. The nearest neighbour scores
1.000 with shuffled rows, because every validation row has a sibling reading of the
same drive in training and copies its label. With each drive's rows kept together it
scores 0.620. Same model, same rows: 0.38 of AUC was the model looking the answer up.

**Try changing.** replace the nearest neighbour by `fresh_model()`: does logistic regression notice
the difference as much?

## 9. Bias, variance and noise at 30 °C, by hand

The lab of slides 26 to 30: 300 training sets of 25 days, each drawn afresh from the
curve we wrote, a straight line fitted to each, and every line asked for one new day at
30 °C. The truth there is known, so the 300 predictions split into bias², variance and
noise.

*From:* slides *The bias–variance decomposition: expected error = bias² + variance +
noise* and *The straight line (degree 1) at 30 °C, in three numbers*; notebook 2,
section 2.

### Your prediction, before you run the next cell

Will the average of the 300 predictions sit above or below the true 330.5 kWh? Which
will be larger at 30 °C, bias² or variance? And does the truth enter the variance?

In [14]:
rng = np.random.default_rng(0)                    # notebook 2's seed
predictions = []
for _ in range(300):
    days_t, days_e = sample_energy(25, rng)       # a fresh training set of 25 days
    straight = make_pipeline(PolynomialFeatures(1), LinearRegression())
    predictions.append(straight.fit(days_t.reshape(-1, 1), days_e).predict([[30.0]])[0])
predictions = np.array(predictions)

truth = float(true_curve(30.0))
average = predictions.sum() / 300
bias_squared = (average - truth) ** 2
variance = np.sum((predictions - average) ** 2) / 300    # square each term, then sum
noise = 22.0 ** 2
print(f"truth f(30)                    {truth:8.1f} kWh")
print(f"the 300 predictions            {predictions[0]:.1f}, {predictions[1]:.1f}, "
      f"{predictions[2]:.1f}, ... from {predictions.min():.1f} to {predictions.max():.1f} kWh")
print(f"average = sum / 300            {average:8.1f} kWh")
print(f"bias = average - truth         {average - truth:+8.1f} kWh")
print(f"bias²                          {bias_squared:8.1f} kWh²")
print(f"variance                       {variance:8.1f} kWh²")
print(f"noise = 22²                    {noise:8.1f} kWh²")
total = bias_squared + variance + noise
print(f"expected error                 {total:8,.1f} kWh², a typical miss of "
      f"√{total:,.1f} = {np.sqrt(total):.0f} kWh")

truth f(30)                       330.5 kWh
the 300 predictions            300.8, 317.1, 304.8, ... from 215.6 to 364.7 kWh
average = sum / 300               303.6 kWh
bias = average - truth            -26.9 kWh
bias²                             723.9 kWh²
variance                          856.1 kWh²
noise = 22²                       484.0 kWh²
expected error                  2,063.9 kWh², a typical miss of √2,063.9 = 45 kWh


**What to look at.** The average of the 300 straight lines at 30 °C is 303.6 kWh,
26.9 below the truth 330.5: bias² = (303.6 − 330.5)² = 723.9. The predictions spread
from 215.6 to 364.7; squared deviations from their own average, summed, divided by 300,
give a variance of 856.1, and the truth does not enter it. With the noise, 22² = 484,
the three make 2,063.9 kWh², a typical miss of about 45 kWh on a new day at 30 °C: the
numbers of slide *The straight line (degree 1) at 30 °C, in three numbers*. Square each
deviation before summing: summed first, the deviations from the average cancel to zero.

**Try changing.** fit degree 2 instead of 1 (`PolynomialFeatures(2)`): which of the
three terms falls, and which does not move?

## 10. Without the truth: training error against validation error

In real data the true curve is unknown, so bias and variance cannot be computed. Two
errors can: the error on the rows a model trained on, and the error on rows it did not
see. `cross_validate(..., return_train_score=True)` returns both for every round.

*From:* slides *Without the truth: two errors you can still measure* and *One sample of 25 days, 5-fold cross-validation*; notebook 2, section 4.

In [15]:
def polynomial(degree):
    return make_pipeline(PolynomialFeatures(degree), LinearRegression())

temperature, energy = sample_energy(25, np.random.default_rng(0))   # 25 days

### Your prediction, before you run the next cell

For degrees 1, 2 and 5 on 25 days: which has the lowest training error, and which
the lowest validation error? Will they be the same degree? The noise alone is
22² = 484 kWh².

In [16]:
print(f"{'degree':>7}{'training MSE':>14}{'validation MSE':>16}{'gap':>9}")
for degree in (1, 2, 5):
    result = cross_validate(polynomial(degree), temperature.reshape(-1, 1), energy,
                            cv=KFold(5, shuffle=True, random_state=0),
                            scoring="neg_mean_squared_error", return_train_score=True)
    train, valid = -result["train_score"].mean(), -result["test_score"].mean()
    print(f"{degree:>7}{train:>14,.1f}{valid:>16,.1f}{valid - train:>9,.1f}")

 degree  training MSE  validation MSE      gap
      1       5,155.7         6,064.3    908.5
      2         312.8           450.4    137.7
      5         262.4           615.1    352.6


**What to look at.** Degree 5 has the lowest training error, 262.4 kWh², below the
noise of 484 that no model can remove: it is fitting the noise. Degree 2 has the lowest
validation error, 450.4: that is the choice. Degree 1 is high on both, 5,155.7 and
6,064.3: too rigid, bias. Degree 5's gap, 352.6, is more than twice degree 2's 137.7:
too flexible, variance. Choosing on the training error would have picked degree 5,
which is why the choice is made on the validation error.

**Try changing.** draw 150 days instead of 25: which gap closes?

## Notebook 3's data, for atoms 11 to 14

Lesson 5's leakage demonstration: the 800 drives' labels, with every feature replaced by
2,000 columns of pure random noise. No model can do better than an AUC of 0.500 on it.
`honest` is slide 45's pipeline, with the selector inside.

In [17]:
noise = pd.DataFrame(np.random.default_rng(0).normal(size=(800, 2_000)),
                     columns=[f"n{i}" for i in range(2_000)])
honest = Pipeline([("select", SelectKBest(f_classif, k=10)),
                   ("scale", StandardScaler()),
                   ("model", LogisticRegression(max_iter=2_000))])
grid = {"select__k": [2, 5, 10, 20, 40], "model__C": [0.01, 0.1, 1, 10, 100]}
inner = StratifiedKFold(5, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)
print(noise.shape, "  labels:", int(y.sum()), "failures")

(800, 2000)   labels: 29 failures


## 11. 0.931 on pure noise, and the fix

The code almost everyone writes: keep the 10 noise columns with the highest F-test,
using all 800 drives, then cross-validate a model on them. The F-test asks, one column
at a time, how far apart the failing and the healthy drives' means are, relative to the
spread inside each group. The fix moves the selector inside the pipeline, `honest`, so
it is refitted in every round on that round's training drives only.

*From:* slides *Now: what cross-validation does not protect you from*, *The code almost
everyone writes*, *0.931* and *Fix it, and look what happens*; notebook 3, sections 1
and 2.

### Your prediction, before you run the next cell

The columns carry no information at all. What AUC will cross-validation report with the
selection done first on all 800 drives: about 0.5, or higher? And with the selection
inside the pipeline?

In [18]:
folds = StratifiedKFold(5, shuffle=True, random_state=0)
selector = SelectKBest(f_classif, k=10).fit(noise, y)      # sees ALL 800 rows and labels
chosen = noise.loc[:, selector.get_support()]
leaky = cross_val_score(fresh_model(), chosen, y, cv=folds, scoring="roc_auc")
print(f"selection first, on all 800: folds {leaky.round(3)}, mean {leaky.mean():.3f}")
inside = cross_val_score(honest, noise, y, cv=folds, scoring="roc_auc")
print(f"selection inside the pipeline: folds {inside.round(3)}, mean {inside.mean():.3f}")
print("the truth: 0.500")

# The F-test made visible: the column the selector liked best.
best = noise.columns[np.argmax(selector.scores_)]
failing, healthy = noise.loc[(y == 1).values, best], noise.loc[(y == 0).values, best]
print(f"\ncolumn {best}: mean over the {len(failing)} failing drives {failing.mean():+.3f}, "
      f"over the {len(healthy)} healthy {healthy.mean():+.3f}")
print(f"a mean of 29 random values wobbles by 1 / √29 = {1 / np.sqrt(29):.2f}; "
      f"this gap is {(failing.mean() - healthy.mean()):.2f}")

selection first, on all 800: folds [0.928 0.979 0.975 0.943 0.832], mean 0.931


selection inside the pipeline: folds [0.671 0.55  0.675 0.688 0.662], mean 0.649
the truth: 0.500

column n438: mean over the 29 failing drives -0.729, over the 771 healthy +0.042
a mean of 29 random values wobbles by 1 / √29 = 0.19; this gap is -0.77


**What to look at.** With the selection done first, on all 800 drives, the five folds
score 0.928 to 0.979 and one 0.832, mean 0.931, on columns that carry nothing: the
selector had seen the labels of every validation fold. Inside the pipeline the mean
falls to 0.649 at this seed (0.658 over 20 seeds on the slide), and not to 0.500: the
columns that match the label by chance match it in all 800 drives, so in every fold.
The F-test made visible: in column n438 the 29 failing drives average −0.729 and the 771
healthy ones +0.042, a gap of 0.77, about four times the 0.19 by which a mean of 29
random values wobbles. Luck, picked out of 2,000 columns.

**Try changing.** set `k=100` in the selector: does the leaky score go up or down, and
the honest one?

## 12. `GridSearchCV`: 25 candidates, the best mean, and a refit nobody scores

`GridSearchCV` tries every combination of the grid, scores each by cross-validation,
and keeps the highest mean. Here **k is the number of columns** `SelectKBest` keeps,
not the number of folds; the folds are fixed at 5 by `cv=inner`. The keys
`"select__k"` and `"model__C"` are the step's name, two underscores, the parameter.

*From:* slides *GridSearchCV: try all, keep the best* and *The best of 25 scores is
inflated: 0.800 for a worthless model*; notebook 3, section 3.

### Your prediction, before you run the next cell

How many fits will the search make, refit included? Will `best_score_` be the highest
of the 25 means, or the AUC of the refitted model?

In [19]:
search = GridSearchCV(honest, grid, cv=inner, scoring="roc_auc").fit(noise, y)
results = pd.DataFrame(search.cv_results_)
means = results.pivot(index="param_select__k", columns="param_model__C",
                      values="mean_test_score")
print("mean AUC over the 5 folds, one cell per candidate (rows k, columns C):")
print(means.round(3).to_string())
print(f"\ncandidates: {len(results)}; fits: {len(results)} × {search.n_splits_} = "
      f"{len(results) * search.n_splits_}, plus 1 refit")
print(f"best_params_: {search.best_params_}")
print(f"best_score_ {search.best_score_:.4f} = the highest mean: "
      f"{np.isclose(search.best_score_, results['mean_test_score'].max())}")
print(f"average candidate {results['mean_test_score'].mean():.4f}, "
      f"worst {results['mean_test_score'].min():.4f}")
winner_rounds = [search.cv_results_[f"split{j}_test_score"][search.best_index_]
                 for j in range(5)]
print(f"the winner's five round AUCs: {' + '.join(f'{v:.3f}' for v in winner_rounds)} "
      f"= {sum(winner_rounds):.3f}; ÷ 5 = {np.mean(winner_rounds):.4f} = best_score_")
print("the refitted model, best_estimator_, keeps",
      search.best_estimator_.named_steps["select"].get_support().sum(), "columns")

mean AUC over the 5 folds, one cell per candidate (rows k, columns C):
param_model__C   0.01    0.10    1.00    10.00   100.00
param_select__k                                        
2                 0.684   0.684   0.685   0.685   0.685
5                 0.708   0.712   0.716   0.717   0.717
10                0.733   0.749   0.759   0.760   0.760
20                0.755   0.780   0.798   0.800   0.798
40                0.696   0.701   0.696   0.692   0.689

candidates: 25; fits: 25 × 5 = 125, plus 1 refit
best_params_: {'model__C': 10, 'select__k': 20}
best_score_ 0.7999 = the highest mean: True
average candidate 0.7265, worst 0.6840
the winner's five round AUCs: 0.750 + 0.754 + 0.879 + 0.740 + 0.877 = 4.000; ÷ 5 = 0.7999 = best_score_
the refitted model, best_estimator_, keeps 20 columns


**What to look at.** 25 candidates, 125 fits, plus one refit. `best_score_`, 0.7999,
is the highest cell of the table (k = 20, C = 10), and it was computed on the folds,
before the refit. The average candidate scores 0.7265 and the worst 0.6840: every cell
is above 0.5 on pure noise, and the best is the luckiest of 25. `best_estimator_` keeps
20 columns, which is what k counts. The refitted model is the one you would ship, and
nobody has scored it: measured on the 800 rows it trained on, it would give a training
score.

**Try changing.** score `search.best_estimator_` on the same 800 rows with `roc_auc_score`: what does
a training score look like on pure noise?

## 13. Nested cross-validation, one outer round at a time

Nested cross-validation runs the whole search inside each outer round. The outer
round sets 160 drives aside; the search runs on the other 640, re-split into five new
inner folds of 128; the winner is refitted on the 640 and scored once on the 160. The
inner scores choose and are thrown away; the outer scores only measure.

*From:* slides *Nested cross-validation: slide 10's recipe, five times* and *Nested cross-validation: it
measures what choosing costs*; notebook 3, section 4.

### Your prediction, before you run the next cell

Will the five rounds pick the same winner? Will each round's inner `best_score_` tell
you what its winner scores on the 160 drives it never saw? And will the mean of the five
outer scores match `cross_val_score(GridSearchCV(...), cv=outer)`?

In [20]:
outer_scores = []
for round_number, (rest, held_out) in enumerate(outer.split(noise, y), 1):
    inner_search = GridSearchCV(honest, grid, cv=inner, scoring="roc_auc").fit(
        noise.iloc[rest], y.iloc[rest])
    score = roc_auc_score(y.iloc[held_out],
                          inner_search.predict_proba(noise.iloc[held_out])[:, 1])
    outer_scores.append(score)
    print(f"round {round_number}: search on {len(rest)} drives, winner "
          f"k = {inner_search.best_params_['select__k']}, "
          f"C = {inner_search.best_params_['model__C']}; inner best "
          f"{inner_search.best_score_:.3f} (thrown away); on the {len(held_out)} set "
          f"aside ({int(y.iloc[held_out].sum())} failures): {score:.3f}")

outer_scores = np.array(outer_scores)
print(f"\nnested: ({' + '.join(f'{v:.3f}' for v in outer_scores)}) ÷ 5 = "
      f"{outer_scores.mean():.3f}")
library = cross_val_score(GridSearchCV(honest, grid, cv=inner, scoring="roc_auc"),
                          noise, y, cv=outer, scoring="roc_auc")
print(f"cross_val_score(GridSearchCV(...), cv=outer): {library.mean():.3f}; "
      f"same five scores: {np.allclose(library, outer_scores)}")

round 1: search on 640 drives, winner k = 10, C = 100; inner best 0.665 (thrown away); on the 160 set aside (5 failures): 0.698


round 2: search on 640 drives, winner k = 20, C = 100; inner best 0.581 (thrown away); on the 160 set aside (6 failures): 0.852


round 3: search on 640 drives, winner k = 10, C = 0.1; inner best 0.593 (thrown away); on the 160 set aside (6 failures): 0.754


round 4: search on 640 drives, winner k = 20, C = 100; inner best 0.692 (thrown away); on the 160 set aside (6 failures): 0.487


round 5: search on 640 drives, winner k = 10, C = 100; inner best 0.710 (thrown away); on the 160 set aside (6 failures): 0.558

nested: (0.698 + 0.852 + 0.754 + 0.487 + 0.558) ÷ 5 = 0.670


cross_val_score(GridSearchCV(...), cv=outer): 0.670; same five scores: True


**What to look at.** The five rounds pick three different winners: k = 10 with C = 100
twice, k = 20 with C = 100 twice, and k = 10 with C = 0.1 once. That is why nested cross-validation does not choose hyperparameters: it
measures what choosing costs. Each search ran on 640 drives, and each outer fold held 5
or 6 failures. The inner `best_score_` is no guide to the outer score: 0.665 against
0.698 in round 1, 0.692 against 0.487 in round 4. On inner folds of 128 drives with four
or five failures each, both numbers are noisy. A maximum is optimistic on average, but
on five rounds the noise can hide it, and the inner score is thrown away either way. The
outer scores run from 0.487 to 0.852; their mean, 3.349 ÷ 5 = 0.670, is exactly what
`cross_val_score` returns for the nested search, 0.13 below the 0.800 the search
reported on all 800.

**Try changing.** change `outer` to `random_state=0`: how far does the nested figure move? Compare
with the five outer scores' own spread.

## 14. A fresh test set: chance correlations in other columns

Even the nested 0.670 stays above the truth, because the columns the selector picks
match the labels of these 800 drives by accident, in every fold. New drives have chance
correlations too, but in other columns: for the chosen ones, the correlation in new rows
is a fresh coin toss.

*From:* slide *Why it does not return to 0.5*; handout section 7.2.

### Your prediction, before you run the next cell

Slide 45's pipeline (k = 10), fitted on the 800 noise rows, is scored on 40 fresh
fleets of 800 drives with new noise. Will the 40 AUCs centre on 0.66, as cross-validation
said, or on 0.5? Can a single one still come out high?

In [21]:
fitted = Pipeline([("select", SelectKBest(f_classif, k=10)),
                   ("scale", StandardScaler()),
                   ("model", LogisticRegression(max_iter=2_000))]).fit(noise, y)
cv_estimate = cross_val_score(honest, noise, y, scoring="roc_auc",
                              cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
fresh = []
for fresh_seed in range(1, 41):
    _, y_new = load_fleet(800, random_state=1000 + fresh_seed)
    X_new = pd.DataFrame(np.random.default_rng(10_000 + fresh_seed).normal(size=(800, 2_000)),
                         columns=noise.columns)
    fresh.append(roc_auc_score(y_new, fitted.predict_proba(X_new)[:, 1]))
fresh = np.array(fresh)
print(f"cross-validation on the 800 (seed 0): {cv_estimate:.3f}")
print(f"40 fresh test sets: mean {fresh.mean():.3f}, from {fresh.min():.3f} "
      f"to {fresh.max():.3f}")

cross-validation on the 800 (seed 0): 0.649
40 fresh test sets: mean 0.509, from 0.401 to 0.680


**What to look at.** Cross-validation on the 800 said 0.649 at this seed (0.658
averaged over 20 seeds, slide *Fix it, and look what happens*). The 40 fresh fleets
average 0.509: centred on the truth, because the columns the model uses match nothing in
new rows. Yet one fresh set gives 0.401 and another 0.680. A test set set aside before
any search is unbiased, not precise, which is why a report states how many failures its
test set held.

**Try changing.** score on fresh sets of 8,000 drives (`load_fleet(8000, ...)` and 8,000 rows of
noise): does the range narrow?

## 15. The threshold: a sweep of 200 candidates on the validation drives

Lesson 4 chose its threshold on the test set. Done properly, on atom 1's split: one model fitted on 450
drives, the threshold chosen on 150 validation drives, the bill reported once on 200
test drives. "Chosen" means a sweep: 200 thresholds from 0.01 to 0.9, each priced at
missed failures × 2,600 EUR + false alarms × 140 EUR, the cheapest kept.

*From:* slide *The debt, paid: the honest threshold costs 1,400 EUR more*; notebook 3, section 5.

In [22]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.25,
                                                  random_state=0, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_rest, y_rest, test_size=0.25,
                                                      random_state=0, stratify=y_rest)
print(f"train {len(X_train)}, validation {len(X_valid)} ({int(y_valid.sum())} failures), "
      f"test {len(X_test)} ({int(y_test.sum())} failures)")
model = fresh_model().fit(X_train, y_train)          # one model, every row below


def bill(y_true, probability, t):
    flagged = probability >= t
    missed = int((~flagged & (y_true == 1)).sum())
    alarms = int((flagged & (y_true == 0)).sum())
    return missed, alarms, missed * 2_600 + alarms * 140


candidates = np.linspace(0.01, 0.9, 200)

train 450, validation 150 (5 failures), test 200 (7 failures)


### Your prediction, before you run the next cell

Will the cheapest threshold on the validation drives be near 0.5, or near lesson 4's
$t^* = 140 / 2{,}740 = 0.051$? On the test set, will it beat the threshold the test set
itself would choose?

In [23]:
valid_p = model.predict_proba(X_valid)[:, 1]
test_p = model.predict_proba(X_test)[:, 1]
costs = [bill(y_valid, valid_p, t)[2] for t in candidates]
chosen = candidates[int(np.argmin(costs))]
print("validation bill at a few candidates:")
for t in candidates[[0, 4, 9, 18, 45, 99, 199]]:
    print(f"  t = {t:.3f}: {bill(y_valid, valid_p, t)[2]:>6,} EUR")
print(f"cheapest on validation: t = {chosen:.3f}")

cheat = candidates[int(np.argmin([bill(y_test, test_p, t)[2] for t in candidates]))]
for name, t in (("the default", 0.5), ("chosen on validation", chosen),
                ("chosen on test", cheat)):
    m, a, c = bill(y_test, test_p, t)
    print(f"{name:>21}, t = {t:.3f}: {m} × 2,600 + {a} × 140 = {c:,} EUR on the test set")

validation bill at a few candidates:
  t = 0.010:  6,720 EUR
  t = 0.028:  3,360 EUR
  t = 0.050:  2,240 EUR
  t = 0.091:  4,420 EUR
  t = 0.211:  2,740 EUR
  t = 0.453:  2,600 EUR
  t = 0.900: 13,000 EUR
cheapest on validation: t = 0.050
          the default, t = 0.500: 4 × 2,600 + 1 × 140 = 10,540 EUR on the test set
 chosen on validation, t = 0.050: 1 × 2,600 + 15 × 140 = 4,700 EUR on the test set
       chosen on test, t = 0.171: 1 × 2,600 + 5 × 140 = 3,300 EUR on the test set


**What to look at.** Train 450, validation 150 with 5 failures, test 200 with 7. The
validation bill is not smooth: 6,720 EUR at 0.010, 2,240 at 0.050, then 4,420 at 0.091
and 2,740 at 0.211, because with five failures every missed one adds 2,600 EUR at once.
The cheapest is 0.050, next to lesson 4's $t^* = 0.051$. On the test set the default
costs 10,540 EUR and the validation's threshold 4,700, the number to report. The test
set's own choice, 0.171, would cost 3,300: cheaper, and not reportable, because it was
chosen by looking at the bill being reported.

**Try changing.** double the cost of a missed failure to 5,200 EUR: where does the validation set's
threshold move?

## 16. Seeds: the same seed, the same number

Every random step draws from a random number generator. `random_state=0` fixes
scikit-learn's draws on the call that receives it; `np.random.default_rng(seed)` makes a
NumPy generator for your own draws. Neither makes the number right: atom 5's spread is
still there.

*From:* slides *A fixed seed buys repeatability, not stability* and *What to fix, and
what to write down* (both optional); notebook 3, section 6.

### Your prediction, before you run the next cell

Two calls of `train_test_split` with `random_state=0`: same test drives? Two calls
with no `random_state`? Two generators made with `default_rng(0)`: same draws?

In [24]:
def test_drives(**seed):
    _, X_te, _, _ = train_test_split(X, y, test_size=0.25, stratify=y, **seed)
    return set(X_te.index)

print("random_state=0 twice, same 200 test drives:",
      test_drives(random_state=0) == test_drives(random_state=0))
print("no random_state twice, same 200 test drives:", test_drives() == test_drives())
print("default_rng(0) twice:", np.random.default_rng(0).normal(size=3).round(3),
      np.random.default_rng(0).normal(size=3).round(3))

aucs = []
for seed in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed,
                                              stratify=y)
    aucs.append(roc_auc_score(y_te, fresh_model().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]))
print(f"\n30 seeds, each reproducible: test AUC from {min(aucs):.3f} to {max(aucs):.3f}")

random_state=0 twice, same 200 test drives: True
no random_state twice, same 200 test drives: False
default_rng(0) twice: [ 0.126 -0.132  0.64 ] [ 0.126 -0.132  0.64 ]

30 seeds, each reproducible: test AUC from 0.913 to 1.000


**What to look at.** The same `random_state` gives the same 200 test drives; no
`random_state` gives different ones on every call; two generators made with
`default_rng(0)` draw the same numbers. Yet 30 seeds, each perfectly reproducible, give
test AUCs from 0.913 to 1.000. A seed makes a number repeatable, not right: write the
seed down, and report the spread too (atom 5).

**Try changing.** find the seed that gives 1.000 and write the two report lines: the misleading one, and
the honest one with the cross-validated mean and spread.